# Data Warehouse

## 1. The problem it solves

A company has many systems:

* **Sales app** (orders), **CRM** (customers), **HR** (employees), **website** (clicks), **Excel files** from finance.

Your manager asks: *“What was revenue per city last quarter, compared to last year?”*

The data is in **5 different places**, in different formats, with different customer IDs. And if you run a heavy report directly on the sales app’s database, **the app slows down for real customers**.

**A data warehouse (DWH) = one central, clean, organised copy of the company’s data, built only for analysis and reporting.**

**Analogy:** shops (source systems) each have their own messy stockroom. A **central warehouse** collects goods from every shop, cleans and labels them, and arranges them on shelves, so anyone can find what they need fast without disturbing the shops.

## 2. OLTP vs OLAP: the most important concept

|               | **OLTP** (source systems)                          | **OLAP** (data warehouse)                              |
| ------------- | -------------------------------------------------- | ------------------------------------------------------ |
| Full form     | Online **Transaction** Processing                  | Online **Analytical** Processing                       |
| Purpose       | **Run** the business                               | **Analyse** the business                               |
| Example       | “Insert this order”, “update Ravi’s address”       | “Total revenue by city for 3 years”                    |
| Queries       | Small, fast, one row at a time                     | Big, scan millions of rows, aggregate                  |
| Users         | Apps, thousands of customers                       | Analysts, BI dashboards, data scientists               |
| Data          | **Current** state only                             | **History** over years                                 |
| Design        | **Normalised** (many small tables, no duplication) | **Denormalised** (star schema, fewer joins)            |
| Storage style | Row-based                                          | Usually **column-based** (fast for SUM/AVG)            |
| Examples      | MySQL, PostgreSQL, Oracle, SQL Server              | Snowflake, Databricks SQL, BigQuery, Redshift, Synapse |

**One-liner:** OLTP is for *writing* transactions, OLAP is for *reading* and analysing them.

## 3. The 4 characteristics (Bill Inmon’s definition, often asked)

A data warehouse is:

1. **Subject-oriented:** organised by business topic (Sales, Customer, Product), not by application.
2. **Integrated:** data from many sources made consistent (one customer ID, one date format, one currency).
3. **Time-variant:** keeps **history**. You can ask “what was it last year?” (This is where your **SCD Type 2** comes in.)
4. **Non-volatile:** data is loaded and read, **not** constantly updated or deleted like in OLTP.

## 4. Architecture: how data flows

```text
 SOURCES                 STAGING                 WAREHOUSE                  CONSUMERS
 ───────                 ───────                 ─────────                  ─────────
 Sales DB  ─┐
 CRM       ─┼──► Extract ──► Staging area ──► Transform + Load ──► Core DWH ──► Data marts ──► BI dashboards
 HR        ─┤              (raw copy)        (clean, join,         (facts +      (Sales mart,   (Power BI, Tableau)
 Files/API ─┘                                 dedupe, history)      dimensions)   Finance mart)  SQL users, ML
```

* **Staging:** a raw landing copy. Like **Bronze**.
* **Core warehouse:** cleaned, integrated, with history. Like **Silver**.
* **Data marts:** small, subject-specific views for one department. Like **Gold**.

So the **Medallion architecture you already know is basically a data warehouse built on a lakehouse.**

## 5. ETL vs ELT

|                     | ETL                                                                      | ELT                                               |
| ------------------- | ------------------------------------------------------------------------ | ------------------------------------------------- |
| Order               | Extract → **Transform** → Load                                           | Extract → **Load** → Transform                    |
| Where it transforms | In a separate tool **before** the warehouse (Informatica, SSIS, **SAS**) | **Inside** the warehouse/lakehouse with SQL/Spark |
| Era                 | Traditional, on-premise                                                  | Modern cloud (Databricks, Snowflake)              |

**Your SAS → PySpark migration is literally ETL → ELT.** SAS did the transforms outside the database; now Spark does them inside Databricks.

## 6. Facts and dimensions (preview of data modelling)

Inside the warehouse, data is split into 2 kinds of tables:

* **Fact table:** **events/measurements**, things you count or sum. Example: `fact_sales` (order_id, date_key, customer_key, product_key, **quantity, amount**). Very many rows.
* **Dimension table:** **descriptions/context**, the “who, what, where, when”. Example: `dim_customer` (name, city), `dim_product` (name, category), `dim_date` (day, month, year). Fewer rows.

```text
            dim_date
               │
dim_customer ─ fact_sales ─ dim_product
               │
            dim_store
```

This is a **star schema**, and it’s the core of data modelling, which we’ll do next in detail.

A query in business language becomes:

> “Revenue (**fact**) by city (**customer dim**) by month (**date dim**).”

## 7. Types of warehouse setups

* **Enterprise Data Warehouse (EDW):** one central warehouse for the whole company.
* **Data mart:** a small, focused warehouse for one team (Sales mart, HR mart).
* **ODS (Operational Data Store):** near-real-time, integrated, current data, but little history. Used for operational reporting.

**Two classic design approaches** (interview favourite):

|             | **Inmon (top-down)**               | **Kimball (bottom-up)**                              |
| ----------- | ---------------------------------- | ---------------------------------------------------- |
| Build first | Central normalised EDW             | Business-process like Sales **star schemas** (data marts)       |
| Then        | Create data marts from it          | Connect marts with **shared (conformed) dimensions** |
| Pros        | Single source of truth, consistent | Faster to deliver, easy for users                    |
| Cons        | Slow and expensive to build        | Risk of inconsistency if not governed                |

Most modern teams use **Kimball-style star schemas**, often in the Gold layer.

## 8. Data warehouse vs data lake vs lakehouse

|             | Data warehouse                               | Data lake                                 | **Lakehouse** (Databricks)       |
| ----------- | -------------------------------------------- | ----------------------------------------- | -------------------------------- |
| Data        | Structured, cleaned                          | Anything: raw files, JSON, images         | Both                             |
| Schema      | Defined **before** loading (schema-on-write) | Defined **when reading** (schema-on-read) | Both, with enforcement           |
| Storage     | Proprietary, costly                          | Cheap object storage (S3, ADLS)           | Cheap storage + **Delta** format |
| Reliability | ACID, fast SQL                               | No ACID, can become a “data swamp”        | **ACID via Delta**, fast SQL     |
| Good for    | BI reports                                   | ML, raw storage                           | BI **and** ML on one copy        |

**Lakehouse = warehouse features (ACID, SQL, performance, governance) on top of a data lake.** In Databricks: **Delta Lake** (ACID, time travel) + **Unity Catalog** (governance) + **Databricks SQL** (warehouse queries).

## 9. Key terms you’ll hear

* **Grain:** what one row in a fact table means (“one row per order line per day - Row 1 = "Order #1001 was placed by customer 1 on Feb 15 for ₹250."”). The most important modelling decision.
* **Surrogate key:** an artificial ID created by the warehouse (`customer_key = 1001`) instead of the source’s ID. Needed for SCD2 history.
* **Natural/business key:** the ID from the source system (`customer_id = C-55`).
* **SCD (Slowly Changing Dimension):** how to handle changes in dimension data (Type 1 overwrite, Type 2 keep history). You already know Type 2.
* **Conformed dimension:** one dimension shared by many facts (the same `dim_date` everywhere).
* **Incremental load:** load only new/changed data each run, not everything. Uses MERGE, watermarks, CDC.
* **Data lineage:** where data came from and how it changed. Unity Catalog shows this.

## 10. A real example end-to-end (retail company)

1. **Sources:** POS system (orders), CRM (customers), product master (Excel).
2. **Bronze/staging:** load raw daily files as-is.
3. **Silver/core:** clean, dedupe, standardise city names, apply SCD2 to customers.
4. **Gold/marts:** `fact_sales` + `dim_customer`, `dim_product`, `dim_date`, `dim_store`.
5. **Consumers:** a Power BI dashboard “Revenue by region by month”, and a data scientist’s churn model.

## Interview one-liners

* “A data warehouse is a central, integrated, historical store of data optimised for analytics (OLAP), separate from transactional systems (OLTP).”
* “We follow Medallion: Bronze = staging, Silver = integrated core, Gold = dimensional star-schema marts.”
* “Modern lakehouses like Databricks give warehouse capabilities on Delta Lake, so BI and ML use one copy of data.”
